## Día 4 · De un 10-K a un resumen verificado

Notebook que acompaña al post del reto *IA aplicada a finanzas*. Hace tres cosas:

1. Descarga el 10-K de Target del año fiscal cerrado el 28 de enero de 2023 y extrae el
   Item 7 (MD&A) como texto, listo para cargarlo en NotebookLM o en cualquier asistente.
2. Revisa cada cifra de un resumen generado por IA y busca si aparece en el documento.
3. Recalcula las cifras que la IA derivó y que no están escritas en la fuente.

**Antes de correrlo:** define la variable de entorno `SEC_USER_AGENT` con tu nombre y
correo, como en el día 1.

In [1]:
import os
import re
import time
from html import unescape
from pathlib import Path

import pandas as pd
import requests

USER_AGENT = os.environ.get("SEC_USER_AGENT", "")
if "@" not in USER_AGENT:
    raise ValueError("Define SEC_USER_AGENT con tu nombre y correo antes de correr el notebook.")
HEADERS = {"User-Agent": USER_AGENT}

DATA = Path("../data")
RAW = DATA / "raw"
RAW.mkdir(parents=True, exist_ok=True)
CIK_TARGET = 27419
CIERRE = "2023-01-28"

## 1. Descargar el 10-K y extraer el Item 7

El endpoint `submissions` de la SEC lista todas las presentaciones de la empresa. Se
busca el 10-K cuyo período cierra el 28 de enero de 2023 y se descarga el documento
principal.

In [2]:
#| label: out-fuente
sub = requests.get(f"https://data.sec.gov/submissions/CIK{CIK_TARGET:010d}.json", headers=HEADERS, timeout=30).json()
rec = pd.DataFrame(sub["filings"]["recent"])
fila = rec[(rec["form"] == "10-K") & (rec["reportDate"] == CIERRE)].iloc[0]
url = (f"https://www.sec.gov/Archives/edgar/data/{CIK_TARGET}/"
       f"{fila['accessionNumber'].replace('-', '')}/{fila['primaryDocument']}")

cache = RAW / "10k_target_2022.htm"
if not cache.exists():
    time.sleep(0.2)
    cache.write_bytes(requests.get(url, headers=HEADERS, timeout=60).content)
texto = re.sub(r"\s+", " ", unescape(re.sub(r"<[^>]+>", " ", cache.read_text(encoding="utf-8", errors="ignore"))))

inicio = re.search(r"Item 7\. Management's Discussion", texto).start()
fin = inicio + re.search(r"Item 7A\. Quantitative", texto[inicio:]).start()
item7 = texto[inicio:fin]
(DATA / "fuente_item7_target_2022.txt").write_text(item7, encoding="utf-8")

print(f"Documento: {url}")
print(f"Presentado el {fila['filingDate']}; Item 7 extraído: {len(item7.split()):,} palabras".replace(",", "."))

Documento: https://www.sec.gov/Archives/edgar/data/27419/000002741923000015/tgt-20230128.htm
Presentado el 2023-03-08; Item 7 extraído: 6.128 palabras


## 2. El verificador de cifras

La función `cifras_resumen` encuentra cada número del resumen (escrito en español) y lo
convierte a una cifra comparable: los porcentajes como porcentaje y los montos en
millones de dólares. La función `cifras_fuente` hace lo mismo con el 10-K (escrito en
inglés), donde los montos aparecen como `109,120` en las tablas o `$4.0 billion` en el
texto. Después se busca cada cifra del resumen en la fuente.

**Lo que verifica y lo que no.** Comprueba que el número exista en el documento, no que
la IA lo haya atribuido al concepto correcto. Por eso muestra también el fragmento donde
lo encontró: la última revisión es humana.

In [3]:
def numero_es(t):
    """'109.120' -> 109120 ; '2,9' -> 2.9 (formato español)."""
    return float(t.replace(".", "").replace(",", "."))

def cifras_resumen(texto):
    cifras = []
    patron = r"(\d{1,3}(?:\.\d{3})+(?:,\d+)?|\d+(?:,\d+)?)(\s*%|\s+mil millones|\s+millones)?"
    for m in re.finditer(patron, texto):
        valor, unidad = m.group(1), (m.group(2) or "").strip()
        if re.fullmatch(r"(19|20)\d\d", valor) or texto[max(0, m.start() - 1)].isalpha():
            continue                                    # años y códigos como A2
        if unidad == "%":
            cifras.append(("porcentaje", numero_es(valor), m.group(0)))
        elif unidad == "mil millones":
            cifras.append(("monto", numero_es(valor) * 1000, m.group(0)))
        elif unidad == "millones" or "." in valor:
            cifras.append(("monto", numero_es(valor), m.group(0)))
    return cifras

def cifras_fuente(texto):
    """Índice {(tipo, valor): fragmento} con las cifras del 10-K (formato inglés)."""
    indice = {}
    patron = r"(\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)\)?(\s*%|\s+percent|\s+billion|\s+million)?"
    for m in re.finditer(patron, texto):
        valor = float(m.group(1).replace(",", ""))
        unidad = (m.group(2) or "").strip()
        contexto = texto[max(0, m.start() - 70): m.end() + 30]
        if unidad == "billion":
            claves = [("monto", round(valor * 1000, 1))]
        elif unidad in ("%", "percent"):
            claves = [("porcentaje", valor)]
        else:                                            # números de tabla, sin unidad escrita
            claves = [("monto", valor), ("porcentaje", valor)]
        for k in claves:
            indice.setdefault(k, contexto)
    return indice

In [4]:
#| label: tbl-verificacion
resumen = (DATA / "resumen_ia.md").read_text(encoding="utf-8")
cuerpo = resumen.split("\n\n", 2)[2]                    # sin título ni nota de procedencia
indice = cifras_fuente(item7)

filas = []
for tipo, valor, original in cifras_resumen(cuerpo):
    contexto = indice.get((tipo, round(valor, 1)))
    filas.append({"Cifra en el resumen": original.strip(), "Tipo": tipo,
                  "¿Está en el 10-K?": "sí" if contexto else "NO",
                  "Fragmento del 10-K": f"…{contexto}…" if contexto else ""})
verificacion = pd.DataFrame(filas)
verificacion.to_csv(DATA / "verificacion_resumen.csv", index=False)
verificacion.style.hide(axis="index").set_properties(subset=["Fragmento del 10-K"], **{"font-size": "0.8em"})

Cifra en el resumen,Tipo,¿Está en el 10-K?,Fragmento del 10-K
"2,9 %",porcentaje,sí,"…sted diluted earnings per share were $6.02. • Total revenue increased 2.9 percent, reflecting total sales growt…"
109.120 millones,monto,sí,"…0 2.8 % 13.2 % Other revenue 1,532 1,394 1,161 9.8 20.2 Total revenue 109,120 106,005 93,561 2.9 13.3 Cost …"
"57,0 %",porcentaje,sí,…d sales increased 1.5 percent. • Operating income of $3.8 billion was 57.0 percent lower than the comparable pri…
3.848 millones,monto,sí,"…cluded in cost of sales) 2,385 2,344 2,230 1.8 5.1 Operating income $ 3,848 $ 8,946 $ 6,539 (57.0) % 36.8…"
"60,0 %",porcentaje,sí,"…2 2021 2020 2022/2021 2021/2020 Net earnings $ 2,780 $ 6,946 $ 4,368 (60.0) % 59.0 % + Provision for income…"
6.946,monto,sí,"…n millions) 2022 2021 2020 2022/2021 2021/2020 Net earnings $ 2,780 $ 6,946 $ 4,368 (60.0) % 59.0 % + Pro…"
2.780 millones,monto,sí,"…ollars in millions) 2022 2021 2020 2022/2021 2021/2020 Net earnings $ 2,780 $ 6,946 $ 4,368 (60.0) % 59.0…"
"6,6 %",porcentaje,NO,
"2,5 %",porcentaje,NO,
"28,3 %",porcentaje,sí,…(57.0) % 36.8 % Rate Analysis 2022 2021 2020 Gross margin rate 23.6 % 28.3 % 28.4 % SG&A expense rate 18.9…


## 3. Las cifras que la IA calculó

Las cifras marcadas con NO no están escritas en el 10-K. No necesariamente están mal:
pueden ser cálculos de la IA. En ese caso hay que rehacer el cálculo con las cifras de
la fuente. Aquí, el margen neto.

In [5]:
#| label: out-derivadas
def cifra_tabla(etiqueta):
    """Primeras dos cifras que siguen a una etiqueta de tabla del Item 7 (2022 y 2021)."""
    m = re.search(re.escape(etiqueta) + r" \$? ?([\d,]+) \$? ?([\d,]+)", item7)
    return float(m.group(1).replace(",", "")), float(m.group(2).replace(",", ""))

utilidad = cifra_tabla("Net earnings")
ingresos = cifra_tabla("Total revenue")
for anio, u, i in zip(["2022", "2021"], utilidad, ingresos):
    print(f"Margen neto {anio}: {u:,.0f} / {i:,.0f} = {u / i * 100:.1f} %")

Margen neto 2022: 2,780 / 109,120 = 2.5 %
Margen neto 2021: 6,946 / 106,005 = 6.6 %
